In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [ ]:
from all_imports import *
import numpy as np
from gpu_warm_up import gpu_warmup
from text_cleaner import TextCleaner
from compute_metrics import compute_metrics
from multimodal_architecture_updated import *
import datasets as hf_datasets

In [ ]:
logging.set_verbosity_error()

In [ ]:
def set_all_seeds(seed_value):
    """
    Sets all relevant random seeds to ensure reproducibility across runs.
    Covers Python's random, NumPy, PyTorch (CPU and CUDA), cuDNN, and
    the HuggingFace Transformers set_seed utility.

    Args:
        seed_value: Integer seed to apply globally.
    """
    random.seed(seed_value)
    np.random.seed(seed_value)
    if torch.cuda.is_available():
        torch.manual_seed(seed_value)
        # torch.cuda.manual_seed_all(seed_value)  # Uncomment for multi-GPU
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.manual_seed(seed_value)
    set_seed(seed_value)  # HuggingFace Transformers seed

In [ ]:
# Uncomment the desired model_name to switch text encoders.

# model_name = "ibm-granite/granite-embedding-30m-english"
# model_name = 'FacebookAI/roberta-base'
# model_name = 'GroNLP/hateBERT'
# model_name = 'google-bert/bert-base-uncased'
# model_name = 'answerdotai/ModernBERT-base'
# model_name = 'sentence-transformers/all-MiniLM-L6-v2'
model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)
many = 50  # Number of Optuna hyperparameter search trials

In [ ]:
# Maps model identifiers to (display_name, pooling_strategy, fusion_layer_sizes).
# Models with hidden_size < 512 start fusion layers at 256.

MODEL_CONFIGS = {
    'google-bert/bert-base-uncased':              ('BaseBERT',       'mean', [512, 256, 128, 64]),
    'GroNLP/hateBERT':                            ('HateBERT',       'mean', [512, 256, 128, 64]),
    'FacebookAI/roberta-base':                    ('RoBERTa',        'mean', [512, 256, 128, 64]),
    'answerdotai/ModernBERT-base':                ('ModernBERT',     'mean', [512, 256, 128, 64]),
    'distilbert-base-uncased':                    ('DistilBERT',     'mean', [512, 256, 128, 64]),
    'ibm-granite/granite-embedding-30m-english':  ('IBM-Granite',    'mean', [256, 128, 64]),  # hidden_size < 512
    'sentence-transformers/all-MiniLM-L6-v2':     ('All-MiniLM-L6', 'mean', [256, 128, 64]),  # hidden_size < 512
}

In [ ]:
# TF32 gives a speedup over FP32 on Ampere GPUs with negligible accuracy loss.

torch.backends.cudnn.conv.fp32_precision = 'tf32'
torch.backends.cuda.matmul.fp32_precision = 'tf32'
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
SEED = 41
set_all_seeds(SEED)
display_name, *_ = MODEL_CONFIGS[model_name]
safe_model_name = display_name.replace(' ', '_')  # Filesystem-safe name for output

In [ ]:
# Train and validation splits are merged since cross-validation handles splitting.

TRAIN_CSV = "../MultiOFF_Dataset/Split Dataset/Training_meme_dataset.csv"
VAL_CSV   = "../MultiOFF_Dataset/Split Dataset/Validation_meme_dataset.csv"
TEST_CSV  = "../MultiOFF_Dataset/Split Dataset/Testing_meme_dataset.csv"
IMG_DIR   = "../Multimodal Project/MultiOFF_Dataset/Labelled Images/"

df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)
df_val   = pd.read_csv(VAL_CSV)

df_train_val = pd.concat([df_train, df_val], ignore_index=True)

In [ ]:
label_encoder = LabelEncoder()
df_train_val["label"] = label_encoder.fit_transform(df_train_val["label"])
df_train['label'] = label_encoder.transform(df_train['label'].tolist())
df_test['label'] = label_encoder.transform(df_test['label'].tolist())
df_val['label'] = label_encoder.transform(df_val['label'].tolist())

In [ ]:
# Applies the project's custom TextCleaner to normalize meme text
# (e.g., lowercasing, punctuation handling, URL removal).

cleaner = TextCleaner()
df_train_val['sentence'] = df_train_val['sentence'].apply(cleaner.clean_text)
df_train['sentence']     = df_train['sentence'].apply(cleaner.clean_text)
df_test['sentence']      = df_test['sentence'].apply(cleaner.clean_text)
df_val['sentence']       = df_val['sentence'].apply(cleaner.clean_text)

In [ ]:
# Computes balanced class weights from the combined train+val set and converts
# them to a tensor for use in the weighted CrossEntropyLoss during training.

from sklearn.utils.class_weight import compute_class_weight

train_labels = df_train_val['label'].tolist()

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

train_class_weights = torch.tensor(class_weights, dtype=torch.float32)

In [ ]:
def get_token_lengths(texts, tokenizer):
    """
    Computes the number of tokens per text sample (including special tokens,
    without truncation) to inform the max_length hyperparameter.

    Args:
        texts:     List of raw text strings.
        tokenizer: HuggingFace tokenizer to use for encoding.

    Returns:
        NumPy array of token counts, one per input text.
    """
    lengths = []
    for t in texts:
        enc = tokenizer(
            t,
            truncation=False,
            padding=False,
            add_special_tokens=True
        )
        lengths.append(len(enc["input_ids"]))
    return np.array(lengths)

lengths = get_token_lengths(df_train_val["sentence"].tolist(), tokenizer)


In [ ]:
# Visualizes the token length distribution of the training set with
# percentile markers to guide max_length selection.

import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(10, 6))

sns.histplot(lengths, bins=50, kde=True, color="skyblue", edgecolor="black", alpha=0.7)

percentile_90  = np.percentile(lengths, 90)
percentile_95  = np.percentile(lengths, 95)
percentile_99  = np.percentile(lengths, 99)
percentile_100 = np.percentile(lengths, 100)

plt.axvline(percentile_90,  color='orange', linestyle='--', linewidth=2, label=f'90th Percentile: {percentile_90:.0f}')
plt.axvline(percentile_95,  color='red',    linestyle='--', linewidth=2, label=f'95th Percentile: {percentile_95:.0f}')
plt.axvline(percentile_99,  color='green',  linestyle='--', linewidth=2, label=f'99th Percentile: {percentile_99:.0f}')
plt.axvline(percentile_100, color='purple', linestyle='--', linewidth=2, label=f'100th Percentile: {percentile_100:.0f}')

plt.title(f"MultiOFF {safe_model_name} Token Length Distribution on Training Data", fontsize=13, fontweight='bold', pad=20)
plt.xlabel("Number of Tokens", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)
plt.ylim(0, 100)
plt.legend(title="Statistics", loc='upper left', bbox_to_anchor=(1, 1), frameon=True)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Sets max_token_length based on the 99th percentile of training token lengths.
# Caps sequences to cover 99% of samples without over-padding.
max_token_length = 512  # Default if percentile_99 > 256 or no branch matches

if percentile_99 > 256:
    max_token_length = 512
elif percentile_99 > 128:
    max_token_length = 256
elif percentile_99 > 64:
    max_token_length = 128
elif percentile_99 > 32:
    max_token_length = 64
elif percentile_99 > 16:
    max_token_length = 32

print(max_token_length)

In [ ]:
def tokenize_function(batch):
    """Tokenizes a batch of text samples with padding and truncation."""
    return tokenizer(batch["sentence"], padding=True, truncation=True, max_length=max_token_length)

In [ ]:
# Image features were extracted offline using EfficientNetV2L at 380x380.
# Loaded from a compressed .npz archive to avoid re-running the image encoder

IMG_NAME = "EfficientNetV2L"

feature_file = f"features_{IMG_NAME}.npz"
image_features = np.load(feature_file)["features"]

In [ ]:
# Assigns a fold index (0-4) to each training sample using StratifiedKFold,
# preserving label distribution across folds. Fold IDs are stored directly
# in the dataframe so filtering by fold is straightforward during cross-validation.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

folds = np.zeros(len(df_train_val), dtype=int)

for fold, (_, val_idx) in enumerate(skf.split(df_train_val, df_train_val["label"])):
    folds[val_idx] = fold

df_train_val["fold"] = folds

In [ ]:
# Attaches the pre-extracted image feature vectors to both splits, converts
# them to HuggingFace Datasets, tokenizes text, and sets the PyTorch format
# so the data collator receives tensors directly.

df_train_val['image_features'] = [image_features[i] for i in df_train_val.index]
df_test['image_features']      = [image_features[i] for i in df_test.index]

train_dataset = hf_datasets.Dataset.from_pandas(df_train_val)
test_dataset  = hf_datasets.Dataset.from_pandas(df_test)

tokenized_test  = test_dataset.map(tokenize_function, batched=True)
tokenized_train = train_dataset.map(tokenize_function, batched=True)

# Guard against Dataset.map dropping the image_features column on some versions.
if 'image_features' not in tokenized_train.column_names:
    tokenized_train = tokenized_train.add_column("image_features", df_train_val['image_features'].tolist())
    tokenized_test  = tokenized_test.add_column("image_features", df_test['image_features'].tolist())

tokenized_train.set_format(type='torch', columns=['input_ids', 'attention_mask', 'image_features', 'label'])
tokenized_test.set_format(type='torch',  columns=['input_ids', 'attention_mask', 'image_features', 'label'])

In [ ]:
def lora_train_with_parameters_optuna(params, train_ds, eval_ds):
    model = LoRAMultiModalClassifier(
        num_labels=num_labels,
        image_feature_size=1280,
        model_name=model_name,
        class_weights=train_class_weights,
    )
    model.to(device)

    training_args = TrainingArguments(
        output_dir="./temp_results_copy",
        learning_rate=params['learning_rate'],
        per_device_train_batch_size=params['batch_size'],
        per_device_eval_batch_size=params['batch_size'],
        lr_scheduler_type=params['lr_decay_type'],
        num_train_epochs=params['num_train_epochs'],
        weight_decay=params['weight_decay'],
        warmup_ratio=params['warmup_ratio'],
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_total_limit=1,
        metric_for_best_model="eval_f1",
        logging_strategy='no',
        report_to="none",
        log_level="error",
        disable_tqdm=True,
        remove_unused_columns=False,
        seed=SEED
    )

    data_collator = MultimodalDataCollator(tokenizer=tokenizer)
    best_epoch_cb = BestEpochCallback()

    trainer = MultimodalTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[best_epoch_cb]
    )

    trainer.train()
    trainer.best_epoch_cb = best_epoch_cb
    return trainer

In [ ]:
def objective(trial: Trial, epsilon=0.5e-6):
    """
    Optuna objective function. Runs 5-fold cross-validation for a sampled
    hyperparameter set and returns the mean validation F1 score across folds.

    Hyperparameter search space:
        learning_rate:       Near 1e-5 (tight range via epsilon).
        batch_size:          2 or 4.
        num_train_epochs:    6 to 10.
        weight_decay:        Near 1e-3.
        activation_function: relu, gelu, tanh, or sigmoid.
        warmup_ratio:        0.1 to 0.2.
        lr_decay_type:       linear or cosine.
        lr_decay_factor:     0.01 to 0.5 (log scale).

    Args:
        trial:   Optuna Trial object for parameter sampling.
        epsilon: Half-width of the tight range around learning_rate and weight_decay.

    Returns:
        Mean F1 across the 5 folds, or 0.0 on failure.
    """
    global all_results_optuna

    suggested_params = {
        'learning_rate':        trial.suggest_float('learning_rate', 1e-5 - epsilon, 1e-5 + epsilon, log=True),
        'batch_size':           trial.suggest_categorical('batch_size', [2, 4]),
        'num_train_epochs':     trial.suggest_int('num_train_epochs', 6, 10),
        'weight_decay':         trial.suggest_float('weight_decay', 1e-3 - (epsilon * 100), 1e-3 + (epsilon * 100), log=True),
        'warmup_ratio':         trial.suggest_float('warmup_ratio', 0.1, 0.2),
        'lr_decay_type':        trial.suggest_categorical('lr_decay_type', ['linear', 'cosine']),
        'lr_decay_factor':      trial.suggest_float('lr_decay_factor', 0.01, 0.5, log=True),
    }

    fold_f1s    = []
    fold_losses = []
    cv_fold_count = 5

    try:
        for fold_id in range(cv_fold_count):
            train_fold = tokenized_train.filter(lambda x: x['fold'] != fold_id, keep_in_memory=True)
            val_fold   = tokenized_train.filter(lambda x: x['fold'] == fold_id,  keep_in_memory=True)

            trainer  = lora_train_with_parameters_optuna(suggested_params, train_fold, val_fold)
            callback = getattr(trainer, "best_epoch_cb", None)
            fold_f1s.append(callback.best_f1)
            fold_losses.append(getattr(callback, "last_loss", float("inf")))

            del trainer
            gc.collect()
            torch.cuda.empty_cache()

        avg_f1   = np.mean(fold_f1s)
        avg_loss = np.mean(fold_losses)

        result_entry = suggested_params.copy()
        result_entry.update({'avg_f1': avg_f1, 'avg_loss': avg_loss})
        all_results_optuna.append(result_entry)

        print(f"{Fore.GREEN}Trial {trial.number} finished | CV Avg F1: {avg_f1:.4f} | CV Avg Loss: {avg_loss:.4f}{Style.RESET_ALL}")
        return avg_f1

    except Exception as e:
        print(f"Error in Trial {trial.number}: {e}")
        return 0.0

In [ ]:
# MedianPruner stops unpromising trials early based on intermediate F1 scores.
# Pruning begins after 4 trials have finished and 4 epochs have elapsed per trial.

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=4,
    interval_steps=1
)

# Maximizes macro F1 across trials.
study = optuna.create_study(direction="maximize", pruner=pruner)
print(safe_model_name)

In [ ]:
# Global list to store results
all_results_optuna = []

# Start Timer
start_time = time.perf_counter()

# Run the optimization for a specified number of trials
study.optimize(objective, n_trials=many, show_progress_bar=True) # Run n_trials

# End Timer
end_time = time.perf_counter()
elapsed_time = end_time - start_time
    
# Print Elapsed Time 
print(f"\n\n{Style.BRIGHT}{Fore.BLUE}Training Completed!{Style.RESET_ALL}")
# Format the time to hours, minutes, and seconds for readability
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Training Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")

training_time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"  # Format training time

In [ ]:
best_trial  = study.best_trial
best_params = best_trial.params


display(best_trial)

In [ ]:
row = {
    'trial_number':    best_trial.number,
    'validation_loss': best_trial.value,
    **best_trial.params
}

df_params = pd.DataFrame([row])
df_params.to_csv(
    f'Results CSV/Parameters/{safe_model_name}_best_params.csv',
    index=False
)

In [ ]:
# Retrains on the full training set (all folds) using the best Optuna params,
# then evaluates on the held-out test set.

print(f"{Style.BRIGHT}{Fore.MAGENTA}Using SEED: {SEED}{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.YELLOW}\n{'='*50} {Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.GREEN}Optuna Best Trial Results:{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.YELLOW}{'='*50} {Style.RESET_ALL}")

print(f"\n{Style.BRIGHT}{Fore.YELLOW}{'='*55} ")
print(f"{Style.BRIGHT}{Fore.LIGHTGREEN_EX}FINAL EVALUATION ON TRAINING SET WITH BEST PARAMETERS{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.YELLOW}{'='*55} {Style.RESET_ALL}")

start_time = time.perf_counter()
final_best_trainer = train_with_parameters_optuna(best_params, tokenized_train, tokenized_test)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"\n\n{Style.BRIGHT}{Fore.BLUE}Training Completed!{Style.RESET_ALL}")
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Training Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")
re_training_time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"

test_metrics = final_best_trainer.evaluate(tokenized_test)
torch.cuda.empty_cache()

print(f"\n{Fore.CYAN}TESTING SET METRICS (Best Optuna Model):{Style.RESET_ALL}")
for key, value in test_metrics.items():
    print(f"{Fore.CYAN}{key}: {value:.4f}{Style.RESET_ALL}")

In [ ]:
# Saves all trial results and the final test evaluation to CSV.
# Evaluation CSV appends if it already exists (supports multi-run accumulation).
# Only saves if many > 49 (i.e., a full 50-trial run).

safe_img_name      = re.sub(r'[\\/*?:"<>|/]', '_', IMG_NAME)
df_optuna_results  = pd.DataFrame(all_results_optuna)
print("\nAll Optuna Search Results:")
display(df_optuna_results)

today          = date.today()
formatted_date = today.strftime("%Y-%m-%d")

evaluation_results = {
    "Model_Name":            model_name,
    "Image_Name":            IMG_NAME,
    "Loss":                  test_metrics["eval_loss"],
    "Accuracy":              test_metrics["eval_accuracy"],
    "Precision":             test_metrics["eval_precision"],
    "Recall":                test_metrics["eval_recall"],
    "F1":                    test_metrics["eval_f1"],
    "AUC_PR":                test_metrics["eval_auc_pr"],
    "Epoch":                 test_metrics["epoch"],
    "Parameter Tuning Time": training_time_str,
    "Final Training Time":   re_training_time_str,
    "Date":                  formatted_date
}

df_evaluation_results = pd.DataFrame([evaluation_results])

if many > 49:
    evaluation_file_name = f"{safe_model_name}_{safe_img_name}_evaluation_results.csv"

    if os.path.exists(evaluation_file_name):
        df_evaluation_results.to_csv(evaluation_file_name, mode='a', header=False, index=False)
        print(f"\nAppended evaluation results to {evaluation_file_name}")
    else:
        df_evaluation_results.to_csv(evaluation_file_name, mode='w', header=True, index=False)
        print(f"\nCreated new evaluation file: {evaluation_file_name}")

    df_optuna_results.to_csv(
        f"Results CSV/50 Trials/{safe_model_name}_{safe_img_name}_{many}_trials.csv",
        mode='w', header=True, index=False
    )

In [ ]:
# Setup
today = date.today().strftime("%Y-%m-%d")
safe_img_name = re.sub(r'[\\/*?:"<>|/]', '_', IMG_NAME)
output_dir = "Results CSV/LoRA"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/{safe_model_name}_30_trials.csv"

all_run_results = []

In [ ]:
# Re-trains using the best Optuna parameters 30 times to estimate metric variance.
# Results are aggregated into all_run_results and saved to a single CSV.

print(f"\n{Style.BRIGHT}{Fore.YELLOW}{'='*55} ")
print(f"{Style.BRIGHT}{Fore.LIGHTGREEN_EX}FINAL EVALUATION ON TRAINING SET WITH BEST PARAMETERS{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.YELLOW}{'='*55} {Style.RESET_ALL}")

for i in tqdm(range(1, 31), desc=f"30 Trials -- {display_name}", unit="run"):
    print(f"{Style.BRIGHT}{Fore.CYAN}\n{'='*30}")
    print(f"RUN {i}/30")
    print(f"{'='*30}{Style.RESET_ALL}")

    start_time = time.perf_counter()
    final_best_trainer = lora_train_with_parameters_optuna(
        best_params,
        tokenized_train,
        tokenized_test
    )
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    hours, remainder = divmod(elapsed_time, 3600)
    minutes, seconds = divmod(remainder, 60)
    print(f"{Style.BRIGHT}{Fore.BLUE}Training Completed!{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Training Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")
    re_training_time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}"

    test_metrics = final_best_trainer.evaluate(tokenized_test)
    torch.cuda.empty_cache()

    print(f"{Fore.CYAN}TESTING SET METRICS (Best Optuna Model):{Style.RESET_ALL}")
    for key, value in test_metrics.items():
        print(f"{Fore.CYAN}{key}: {value:.4f}{Style.RESET_ALL}")

    run_data = {
        "Run":                   i,
        "Model_Name":            model_name,
        "Image_Name":            IMG_NAME,
        "Loss":                  test_metrics["eval_loss"],
        "Accuracy":              test_metrics["eval_accuracy"],
        "Precision":             test_metrics["eval_precision"],
        "Recall":                test_metrics["eval_recall"],
        "F1-Score":              test_metrics["eval_f1"],
        "AUC_PR":                test_metrics["eval_auc_pr"],
        "Parameter_Tuning_Time": training_time_str,
        "Final_Training_Time":   re_training_time_str,
        "Date":                  today,
    }

    all_run_results.append(run_data)

    del final_best_trainer, test_metrics, run_data
    gc.collect()
    torch.cuda.empty_cache()

df_30_trials = pd.DataFrame(all_run_results)
df_30_trials.to_csv(output_file, index=False)
print(f"\n{Style.BRIGHT}{Fore.GREEN}Saved 30 trial results to: {output_file}{Style.RESET_ALL}")

In [ ]:
# Summary stats
print(f"\n{Style.BRIGHT}{Fore.YELLOW}{'='*55}")
print(f"SUMMARY ACROSS 30 RUNS FOR {safe_model_name}")
print(f"{'='*55}{Style.RESET_ALL}")
for metric in ["F1-Score", "Accuracy", "Precision", "Recall", "AUC_PR", "Loss"]:
    col = df_30_trials[metric]
    print(f"{Fore.CYAN}{metric:12s} — Mean: {col.mean():.4f}  Std: {col.std():.4f}  Min: {col.min():.4f}  Max: {col.max():.4f}{Style.RESET_ALL}")

In [ ]:
# watch -n 0.1 nvidia-smi